# CNN para Classificação de Pneumonia (Normal, Viral, Bacteriana)

Este notebook treina uma rede neural convolucional (CNN) com base no dataset de radiografias de tórax disponível no Kaggle.

**Classes:**
- NORMAL
- PNEUMONIA_VIRAL
- PNEUMONIA_BACTERIAL

In [1]:
# Imports
import os
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Dataset personalizado
class ChestXRayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.root_dir = root_dir

        for label_name in os.listdir(root_dir):
            label_dir = os.path.join(root_dir, label_name)
            if not os.path.isdir(label_dir):
                continue

            for img_name in os.listdir(label_dir):
                img_path = os.path.join(label_dir, img_name)
                if "NORMAL" in label_name:
                    label = 0
                elif "PNEUMONIA" in label_name:
                    if "bacteria" in img_name:
                        label = 2
                    elif "virus" in img_name:
                        label = 1
                    else:
                        continue
                else:
                    continue
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [3]:
# Transformações
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Diretório base (modifique o caminho para seu dataset)
data_dir = './chest_xray'

# Loaders
train_dataset = ChestXRayDataset(os.path.join(data_dir, 'train'), transform=transform)
val_dataset = ChestXRayDataset(os.path.join(data_dir, 'val'), transform=transform)
test_dataset = ChestXRayDataset(os.path.join(data_dir, 'test'), transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [4]:
# CNN simples
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier = nn.Linear(128, 3)

    def forward(self, x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [5]:
# Treinamento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Train Acc: {acc:.4f}")

Epoch 1, Loss: 164.1746, Train Acc: 0.5182
Epoch 2, Loss: 137.0540, Train Acc: 0.6179
Epoch 3, Loss: 131.6850, Train Acc: 0.6281
Epoch 4, Loss: 123.8072, Train Acc: 0.6461
Epoch 5, Loss: 119.3198, Train Acc: 0.6764
Epoch 6, Loss: 115.8571, Train Acc: 0.6837
Epoch 7, Loss: 111.6688, Train Acc: 0.6994
Epoch 8, Loss: 107.5204, Train Acc: 0.7124
Epoch 9, Loss: 104.5495, Train Acc: 0.7349
Epoch 10, Loss: 103.5994, Train Acc: 0.7345


In [7]:
# Avaliação
model.eval()
correct = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        correct += (outputs.argmax(1) == labels).sum().item()

print(f"Test Accuracy: {correct / len(test_loader.dataset):.4f}")

Test Accuracy: 0.6875


In [8]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("Dispositivo atual:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


CUDA disponível: True
Dispositivo atual: NVIDIA GeForce RTX 3070 Ti Laptop GPU
